# Recommender per-request cost

Estimate the per-recommendation infra cost for a two-stage recommender at a
given DAU. Used by the capstone (video recs @ 100M DAU).

In [ ]:
import math

# illustrative prices (USD)
CPU_VCPU_HOUR      = 0.04
GPU_L4_HOUR        = 0.55
FEATURE_READ_PER_M = 0.10     # K/V store read cost per million ops
ANN_READ_PER_M     = 0.20     # vector lookup, hosted

def per_request_cost(
    sessions_per_dau=4,
    recs_per_session=20,
    feature_reads_per_rec=80,    # ~80 features looked up per request
    ranker_ms=40,                # ranker CPU/GPU time
    ranker_on_gpu=True,
    ann_calls_per_rec=1,
    dau=100_000_000,
):
    requests = dau * sessions_per_dau         # one request per session shows recs_per_session items
    daily_recs = requests * recs_per_session
    # feature reads
    feat_reads = requests * feature_reads_per_rec
    feat_usd = feat_reads / 1e6 * FEATURE_READ_PER_M
    # ann calls
    ann = requests * ann_calls_per_rec
    ann_usd = ann / 1e6 * ANN_READ_PER_M
    # ranker compute
    cpu_hours = requests * ranker_ms / 1000 / 3600
    if ranker_on_gpu:
        compute_usd = cpu_hours * GPU_L4_HOUR
    else:
        compute_usd = cpu_hours * CPU_VCPU_HOUR
    daily = feat_usd + ann_usd + compute_usd
    return {
        'daily_requests': requests,
        'daily_recs':     daily_recs,
        'features_usd':   round(feat_usd),
        'ann_usd':        round(ann_usd),
        'compute_usd':    round(compute_usd),
        'daily_usd':      round(daily),
        'monthly_usd':    round(daily * 30),
        'usd_per_1k_recs': round(1000 * daily / daily_recs, 4),
    }

r = per_request_cost()
for k, v in r.items():
    print(f'{k:18s}  {v}')

Levers, in priority order:

1. **Feature read count.** 80 features at 1 read each is fine. 800 at 1 read each is not. Bundle by entity.
2. **Ranker latency.** A 40ms ranker on L4 GPU is 2-3x cheaper per request than the same on CPU.
3. **Cache hit on session-scoped features.** A session-scoped feature can be cached for the session. Free win.
4. **Drop the ranker for cold sessions.** Heuristic ranking for the first impression; full ranker on the next.

See `07-recommendation-systems/` and `15-capstone/`.